Database Connection

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(find_dotenv())

def connect_to_db():
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")

    if not all([database, user, password]):
        print("Σφάλμα: Δεν βρέθηκαν οι απαραίτητες μεταβλητές στο .env!")
        return None

    try:
        connection_uri = f"postgresql://{user}:{password}@{host}:{port}/{database}"
        engine = create_engine(connection_uri)
        with engine.connect() as connection:
            print(f"Επιτυχής σύνδεση στη βάση '{database}' στο host '{host}'!")
        return engine

    except SQLAlchemyError as e:
        print(f"Σφάλμα κατά τη σύνδεση στη ΒΔ: {e}")
        return None


print("Εκκίνηση σύνδεσης με τη βάση...")
engine = connect_to_db()

if engine is None:
    print("Τερματισμός προγράμματος λόγω αποτυχίας σύνδεσης.")
    exit(1)

Εκκίνηση σύνδεσης με τη βάση...
Επιτυχής σύνδεση στη βάση 'thesis_db' στο host 'dell-micro'!


Import data to dataframe

In [2]:
import pandas as pd

query = "SELECT match_id, x_loc, y_loc, distance_to_goal, angle_to_goal, is_goal FROM understat_shots_normalized"
us_shots = pd.read_sql(query, con=engine)


query = "SELECT match_id, x_loc, y_loc, distance_to_goal, angle_to_goal, is_goal FROM statsbomb_shots_normalized"
sb_shots = pd.read_sql(query, con=engine)

Remove overlapping matches

In [3]:
query = "SELECT understat_match_id FROM mapping_overlapping_matches"
overlapping_matches = pd.read_sql_query(query, con=engine)
overlapping_matches.head()

,understat_match_id
0,923
1,662
2,727
3,890
4,644


In [4]:
print(f"Αρχικές εγγραφές us_shots: {len(us_shots)}")
overlapping_matches_list = overlapping_matches['understat_match_id'].unique()

# Αφαίρεση overlapping matches από το us_shots
us_shots = us_shots[~us_shots['match_id'].isin(overlapping_matches_list)].reset_index(drop=True)

print(f"Overlapping matches: {len(overlapping_matches_list)}")
print(f"Εγγραφές μετά την αφαίρεση: {len(us_shots)}")

Αρχικές εγγραφές us_shots: 605446
Overlapping matches: 1576
Εγγραφές μετά την αφαίρεση: 566725


Drop 'match_id' from DataFrames

In [5]:
us_shots.drop(['match_id'], axis=1, inplace=True)
sb_shots.drop(['match_id'], axis=1, inplace=True)

print(f"{list(us_shots)} και {list(sb_shots)}")

['x_loc', 'y_loc', 'distance_to_goal', 'angle_to_goal', 'is_goal'] και ['x_loc', 'y_loc', 'distance_to_goal', 'angle_to_goal', 'is_goal']


Merge 2 dataframes

In [6]:
shots = pd.concat([us_shots, sb_shots], ignore_index=True)
print(f"Μέγεθος Understat: {len(us_shots)}")
print(f"Μέγεθος StatsBomb: {len(sb_shots)}")
print(f"Μέγεθος Συνολικό : {len(shots)}")

Μέγεθος Understat: 566725
Μέγεθος StatsBomb: 55857
Μέγεθος Συνολικό : 622582


In [7]:
shots.dtypes

x_loc               float64
y_loc               float64
distance_to_goal    float64
angle_to_goal       float64
is_goal               int64
dtype: object

Model Training

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

X = shots.drop(columns=['is_goal'])
y = shots['is_goal']

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

model = RandomForestClassifier()
predictions = cross_val_predict(model, X, y, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]

Metrics

In [11]:
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss

print("Default Random Forest settings, 1shot/leaf, overfitting")
print(f"Log Loss: {log_loss(y, predictions):.4f}")
print(f"ROC AUC:  {roc_auc_score(y, predictions):.4f}")
print(f"Brier Score:  {brier_score_loss(y, predictions):.04f}")

Default Random Forest settings, 1shot/leaf, τρομερό overfitting
Log Loss: 1.0335
ROC AUC:  0.6723
Brier Score:  0.0916


Finding the best params

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

model2 = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions = {
        'n_estimators': [100, 200],
        'max_depth': [5, 10, 20],
        'min_samples_split': [100, 200],
        'min_samples_leaf': [50, 100, 200],
        'max_features': ['sqrt', 'log2', 0.5, None]
    },
    scoring = 'neg_log_loss',
    cv = skf,
    n_iter=20,
    random_state = 42,
    verbose=2,
    n_jobs = -1
)

model2.fit(X, y)

print(f"{model2.best_params_}")
# Best Params:
#


Fitting 10 folds for each of 20 candidates, totalling 200 fits
